### Device (GPU)

In [54]:
import torch
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

Using device: cuda


### import important libraries

In [55]:
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import datasets, transforms, models
from torch.utils.data import DataLoader

### Data Transform

In [56]:
transform = transforms.Compose([
    transforms.Resize((224, 224)),

    transforms.Lambda(lambda img: img.convert("RGB")),

    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(10),

    transforms.ToTensor(),

    transforms.Normalize(
        [0.485, 0.456, 0.406],
        [0.229, 0.224, 0.225]
    )
])

### Load Dataset

In [57]:
train_data = datasets.ImageFolder("../data/combined_data/train", transform=transform)
test_data = datasets.ImageFolder("../data/combined_data/test", transform=transform)

train_loader = DataLoader(train_data, batch_size=64, shuffle=True)
test_loader = DataLoader(test_data, batch_size=64)



In [58]:
print(train_data.classes)

['Angry', 'Disgust', 'Fear', 'Happy', 'Neutral', 'Sad', 'Surprise']


### MODEL (RESNET18)

In [59]:

model = models.resnet18(pretrained=True)

# Output = 7 emotions
model.fc = nn.Linear(model.fc.in_features, 7)

model = model.to(device)

### Loss & Optimizer

In [60]:
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)


### Training Loop

In [61]:
epochs = 10

for epoch in range(epochs):
    model.train()
    running_loss = 0

    for images, labels in train_loader:
        images, labels = images.to(device), labels.to(device)

        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        running_loss += loss.item()

    print(f"Epoch {epoch+1}, Loss: {running_loss:.4f}")


Epoch 1, Loss: 727.6489
Epoch 2, Loss: 604.0097
Epoch 3, Loss: 557.5176
Epoch 4, Loss: 519.9918
Epoch 5, Loss: 493.5980
Epoch 6, Loss: 466.3550
Epoch 7, Loss: 436.5849
Epoch 8, Loss: 408.7749
Epoch 9, Loss: 377.8889
Epoch 10, Loss: 351.9378


### Evaluation

In [62]:
model.eval()
correct = 0
total = 0

with torch.no_grad():
    for images, labels in test_loader:
        images, labels = images.to(device), labels.to(device)

        outputs = model(images)
        _, predicted = torch.max(outputs, 1)

        total += labels.size(0)
        correct += (predicted == labels).sum().item()

accuracy = 100 * correct / total
print(f"Test Accuracy: {accuracy:.2f}%")


Test Accuracy: 73.73%



### Save Model

In [63]:
torch.save(model.state_dict(), "../models/face_emotion_model.pth")